# 🏢 مختبر مسكن ثلاثي الأبعاد: تجربة التدريب عبر Google Colab (T4 GPU)
### Maskan3D Lab: 3D Gaussian Splatting Training Pipeline

يقوم هذا الدفتر بتنفيذ مسار عمل كامل لتحويل صور العقارات والشقق إلى جولات ثلاثية الأبعاد تفاعلية بتقنية **3D Gaussian Splatting (Splatfacto)** باستخدام حزمة **Nerfstudio** المفتوحة المصدر والآمنة للاستخدام التجاري.

---
### 🔄 خطوات المسار (Pipeline Steps):
1. **فحص بطاقة الرسوميات (GPU Check):** فحص بطاقة T4 ومكتبات CUDA وإصدار Python.
2. **تثبيت الحزم (Installation):** تثبيت COLMAP و ffmpeg عبر apt، وتثبيت Nerfstudio عبر pip.
3. **رفع وفك ضغط الصور (Unzip Photos):** فك ضغط ملف `photos.zip` مباشرة وبشكل مسطح داخل `/content/data/images`.
4. **استخراج مواضع الكاميرات (SfM / COLMAP):** تشغيل `ns-process-data` مع خيار `--no-gpu` وحساب نسبة تسجيل الكاميرات.
5. **التدريب (Training):** تدريب نموذج `splatfacto` وحفظ المخرجات محلياً في `/content/outputs`.
6. **التصدير والتنزيل (Export & Download):** تصدير نموذج `.ply`، قياس حجمه بالـ MB، وتنزيله تلقائياً إلى جهازك عبر `google.colab.files.download()`.


## ⚙️ إعداد المتغيرات والمسارات الرئيسية (Configuration & Paths)
قم بمراجعة وتعديل المتغيرات أدناه حسب رغبتك قبل بدء التشغيل.

In [ ]:
# ========================================================
# إعدادات المسارات والمتغيرات الرئيسية
# ========================================================
import os
from pathlib import Path

# مسار ملف الصور المضغوط الذي يتم رفعه في لوحة ملفات Colab
ZIP_PHOTOS_PATH = "/content/photos.zip"

# مجلد فك ضغط الصور
IMAGES_DIR = "/content/data/images"

# مسار بديل في حال استخدام فيديو التجول
VIDEO_PATH = "/content/video.mp4"

# مجلد حفظ مخرجات البيانات المعالجة عبر COLMAP
PROCESSED_DATA_DIR = "/content/data/nerfstudio/apartment"

# مجلد حفظ مخرجات ونقاط تفتيش التدريب (محلياً داخل Colab)
TRAIN_OUTPUTS_DIR = "/content/outputs"

# مجلد تصدير الملفات النهائية
EXPORTS_DIR = "/content/exports"

# اسم التجربة وعدد خطوات التدريب
# ملحوظة: للتجارب السريعة أو عدد صور قليل استخدم 3000 إلى 5000 خطوة (تستغرق ~3-5 دقائق)
# للنتائج النهائية لشقة كاملة استخدم 15000 خطوة (تستغرق ~10-12 دقيقة)
EXPERIMENT_NAME = "apartment_splat"
MAX_ITERATIONS = 5000

print(f"📦 Photos Zip Path:       {ZIP_PHOTOS_PATH}")
print(f"📸 Images Target Dir:      {IMAGES_DIR}")
print(f"💾 Train Outputs Dir:      {TRAIN_OUTPUTS_DIR}")
print(f"🎯 Max Iterations:         {MAX_ITERATIONS}")


### 🎛️ معايير وخوارزميات المعالجة (Pipeline Parameters)
تحديد خوارزميات المطابقة ونوع الكاميرا وأداة SfM المستهدفة.

In [ ]:
# ========================================================
# معايير وخوارزميات المعالجة (Pipeline Parameters)
# ========================================================
MATCHING = "exhaustive"      # "exhaustive" for <=100 photos, "sequential" for video frames
SFM_TOOL = "colmap"          # "colmap" or "hloc"
USE_GPU_FLAG = False         # apt COLMAP is CPU-only on Colab
CAMERA_TYPE = "perspective"
VIDEO_FPS = 2

print(f"🔍 Matching Method:   {MATCHING}")
print(f"🛠️  SfM Tool:          {SFM_TOOL}")
print(f"⚡ Use GPU (COLMAP):   {USE_GPU_FLAG}")
print(f"📷 Camera Type:        {CAMERA_TYPE}")
print(f"🎬 Video FPS:          {VIDEO_FPS}")

## 1️⃣ فحص بطاقة الرسوميات وبيئة التشغيل (Check GPU & Environment)
التحقق من بطاقة الرسوميات (Nvidia T4 GPU)، وإصدار مصرف CUDA (nvcc)، وبيئة بايثون و PyTorch.

In [ ]:
# التحقق من مواصفات بطاقة الرسوميات عبر أداة nvidia-smi
!nvidia-smi

# التحقق من إصدار مصرف CUDA (nvcc)
!nvcc --version

# التحقق من إصدار بايثون وبيئة العمل
import sys
import torch

print(f"🐍 Python Version: {sys.version.split()[0]}")
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"⚡ CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🎮 GPU Device Name: {torch.cuda.get_device_name(0)}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"🧠 Total VRAM: {total_mem_gb:.2f} GB")
else:
    print("⚠️ تحذير: لم يتم اكتشاف GPU! يرجى التأكد من تغيير نوع وقت التشغيل في Colab إلى T4 GPU.")


## 2️⃣ تثبيت الحزم والمتطلبات (Install Dependencies)
- تثبيت **COLMAP** و **FFmpeg** عبر مستودعات النظام `apt`.
- ترقية **pip** وتثبيت مكتبة **nerfstudio** (والتي تتضمن مكتبة `gsplat` المفتوحة تجارياً).

In [ ]:
# تحديث الحزم وتثبيت colmap و ffmpeg عبر apt
!apt-get update -y && apt-get install -y colmap ffmpeg

# ترقية مدير الحزم pip
!pip install --upgrade pip

# تثبيت nerfstudio
!pip install nerfstudio


## 3️⃣ رفع وفك ضغط صور الشقة (Unzip Photos Flat to /content/data/images)
> **طريقة الاستخدام:** 
> اسحب ملف `photos.zip` وأفلته في لوحة الملفات الجانبية بـ Colab تحت مجلد `/content/`.
> ستقوم الخلية أدناه بفك ضغط جميع الصور بشكل مسطح (Flat) داخل `/content/data/images` متجاهلة أي مجلدات فرعية أو ملفات مخفية، ثم طباعة عدد الصور المستخرجة.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

target_dir = Path(IMAGES_DIR)
target_dir.mkdir(parents=True, exist_ok=True)
valid_extensions = {".jpg", ".jpeg", ".png", ".webp"}

zip_path = Path(ZIP_PHOTOS_PATH)

# 1. إذا كان ملف photos.zip موجوداً، فك ضغطه
if zip_path.exists():
    print(f"📦 تم العثور على الأرشيف: {zip_path}، جاري فك الضغط...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            if member.is_dir():
                continue
            filename = os.path.basename(member.filename)
            if not filename or filename.startswith(".") or "__MACOSX" in member.filename:
                continue
            ext = os.path.splitext(filename)[1].lower()
            if ext in valid_extensions:
                dest_path = target_dir / filename
                with zf.open(member) as source, open(dest_path, "wb") as target:
                    target.write(source.read())

# 2. إذا كانت هناك صور مفردة مرفوعة مباشرة في /content أو مسار فرعي
content_dir = Path("/content")
for f in content_dir.glob("*"):
    if f.is_file() and f.suffix.lower() in valid_extensions:
        shutil.copy2(f, target_dir / f.name)

# 3. إحصاء الصور الموجودة في مجلد الهدف
existing_images = [p for p in target_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_extensions]
extracted_count = len(existing_images)

print("=" * 50)
print(f"📂 مجلد الصور المستهدف: {target_dir}")
print(f"📸 إجمالي الصور الجاهزة: {extracted_count}")
print("=" * 50)

if extracted_count == 0:
    print("⚠️ لم يتم العثور على أي صور داخل /content/data/images!")
    print("يرجى سحب ملف photos.zip أو سحب الصور مباشرة إلى لوحة ملفات Colab الجانبية.")
else:
    print("✅ الصور جاهزة للمرحلة التالية.")

## 3️⃣.1 فحص بيانات الكاميرا وجودة الصور (EXIF & Metadata Validation)
فحص دقة الصور ووجود بيانات EXIF الهامة (مثل البعد البؤري FocalLength والشركة المصنعة وطراز الكاميرا) لضمان أعلى دقة لـ COLMAP.

In [ ]:
from pathlib import Path
from PIL import Image, ExifTags

images_path = Path(IMAGES_DIR)
valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
image_files = [p for p in images_path.iterdir() if p.is_file() and p.suffix.lower() in valid_exts] if images_path.exists() else []

total_images = len(image_files)
print("=" * 60)
print(f"🔍 فحص بيانات EXIF وجودة الصور ({total_images} صورة)")
print("=" * 60)

if total_images == 0:
    print("⚠️ لم يتم العثور على أي صور في مجلد الصور للتحقق منها.")
else:
    widths = []
    heights = []
    has_focal_count = 0
    has_make_count = 0
    has_model_count = 0
    lacks_exif_count = 0

    TAG_NAME_TO_ID = {name: val for val, name in ExifTags.TAGS.items()}
    focal_id = TAG_NAME_TO_ID.get("FocalLength", 37386)
    make_id = TAG_NAME_TO_ID.get("Make", 271)
    model_id = TAG_NAME_TO_ID.get("Model", 272)

    for img_file in image_files:
        try:
            with Image.open(img_file) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
                exif = img.getexif()
                exif_dict = dict(exif) if exif else {}
                if hasattr(exif, "get_ifd"):
                    try:
                        exif_ifd = exif.get_ifd(0x8769)
                        if exif_ifd:
                            exif_dict.update(exif_ifd)
                    except Exception:
                        pass

                has_focal = focal_id in exif_dict
                has_make = make_id in exif_dict
                has_model = model_id in exif_dict

                if has_focal:
                    has_focal_count += 1
                if has_make:
                    has_make_count += 1
                if has_model:
                    has_model_count += 1

                if not (has_focal or has_make or has_model):
                    lacks_exif_count += 1
        except Exception:
            lacks_exif_count += 1

    min_res = f"{min(widths)}x{min(heights)}"
    max_res = f"{max(widths)}x{max(heights)}"
    print(f"📸 إجمالي الصور المفحوصة:             {total_images}")
    print(f"📐 نطاق الدقة (Resolution Range):       {min_res} إلى {max_res}")
    print(f"📷 تحتوي على Make (الشركة المصنعة):     {has_make_count} / {total_images}")
    print(f"📱 تحتوي على Model (طراز الكاميرا):     {has_model_count} / {total_images}")
    print(f"🔍 تحتوي على FocalLength (البعد البؤري): {has_focal_count} / {total_images}")

    missing_pct = (lacks_exif_count / total_images) * 100
    print(f"⚠️ نسبة الصور بدون بيانات EXIF:         {missing_pct:.1f}%")
    print("=" * 60)

    if missing_pct > 30.0:
        print(f"⚠️ تحذير: {missing_pct:.1f}% من الصور تفتقر لبيانات EXIF (البعد البؤري، الشركة المصنعة، الطراز)!")
        print("السبب المرجح: تم إرسال الصور عبر Telegram أو WhatsApp كـ 'صورة عادية' مما أدى لضغطها وحذف الـ EXIF.")
        print("💡 نصيحة: يُفضل إعادة رفع الصور الأصلية عبر كابل USB أو رفعها كملف (File) في Google Drive دون ضغط لضمان دقة الحسابات الهندسية لـ COLMAP.")
    else:
        print("✅ بيانات EXIF متوفرة بنسبة جيدة في معظم الصور.")

## 4️⃣ استخراج مواضع الكاميرات من الصور عبر COLMAP (Process Images)
تشغيل أمر `ns-process-data images`.
> **ملاحظة تقنية:** نستخدم الخيار `--no-gpu` لأن حزمة COLMAP المثبتة عبر `apt` تعمل على الـ CPU فقط بدون دعم CUDA/SiftGPU.

In [ ]:
import shutil
from pathlib import Path

# حذف مجلد المخرجات السابق لضمان عدم اختلاط النتائج القديمة
output_path = Path(PROCESSED_DATA_DIR)
if output_path.exists():
    shutil.rmtree(output_path)

gpu_flag = "" if USE_GPU_FLAG else "--no-gpu"

# تشغيل معالجة الصور واستخراج وضعيات الكاميرا
!ns-process-data images \
    --data "{IMAGES_DIR}" \
    --output-dir "{PROCESSED_DATA_DIR}" \
    --camera-type {CAMERA_TYPE} \
    --matching-method {MATCHING} \
    --sfm-tool {SFM_TOOL} \
    {gpu_flag}

## 🎦 [مسار بديل] استخراج إطارات الفيديو ومعالجتها (Video Frame Extraction & Processing)
إذا كنت تستخدم فيديو للتجول في الشقة (`/content/video.mp4`)، قم باستخراج الإطارات عبر الخلية (A) ثم معالجتها عبر الخلية (B).

In [ ]:
# 🎬 الخلية A: استخراج الإطارات من الفيديو بمعدل VIDEO_FPS عبر ffmpeg
import os
import shutil
import subprocess
from pathlib import Path

video_file = Path(VIDEO_PATH)
images_target = Path(IMAGES_DIR)

if not video_file.exists():
    print(f"❌ لم يتم العثور على ملف الفيديو: {video_file}")
    print("يرجى رفع ملف video.mp4 في لوحة ملفات Colab تحت /content/ لاستخدام مسار الفيديو.")
else:
    print(f"🎥 جاري استخراج الإطارات من الفيديو: {video_file}")
    # مسح مجلد الصور أولاً لضمان عدم تداخل إطارات قديمة
    if images_target.exists():
        shutil.rmtree(images_target)
    images_target.mkdir(parents=True, exist_ok=True)

    # استخراج الإطارات باستخدام ffmpeg بمعدل VIDEO_FPS
    ffmpeg_cmd = f'ffmpeg -y -i "{video_file}" -vf fps={VIDEO_FPS} -q:v 2 "{images_target}/frame_%05d.jpg"'
    subprocess.run(ffmpeg_cmd, shell=True, check=True)

    extracted_frames = list(images_target.glob("*.jpg"))
    frame_count = len(extracted_frames)

    print("=" * 60)
    print(f"🎬 تم استخراج الإطارات بنجاح إلى: {images_target}")
    print(f"📸 إجمالي عدد الإطارات المستخرجة: {frame_count}")
    print("=" * 60)

    if frame_count < 80:
        print(f"⚠️ تحذير: عدد الإطارات المستخرجة قليل جداً ({frame_count} < 80 إطار)!")
        print("قد لا تكفي لبناء نموذج ثلاثي الأبعاد متماسك. يُنصح بزيادة مدة الفيديو أو رفع VIDEO_FPS.")
    elif frame_count > 400:
        print(f"⚠️ تحذير: عدد الإطارات كبير جداً ({frame_count} > 400 إطار)!")
        print("قد يستغرق COLMAP وقتاً طويلاً جداً في المعالجة على الـ CPU. يُنصح بتقليل مدة الفيديو أو خفض VIDEO_FPS.")
    else:
        print(f"✅ عدد الإطارات مثالي ({frame_count} إطار ضمن النطاق المطلوب 80-400).")

In [ ]:
# 🚀 الخلية B: معالجة إطارات الفيديو المستخرجة بمطابقة متسلسلة (sequential)
import shutil
from pathlib import Path

# حذف مجلد المخرجات السابق لضمان عدم اختلاط النتائج القديمة
output_path = Path(PROCESSED_DATA_DIR)
if output_path.exists():
    shutil.rmtree(output_path)

gpu_flag = "" if USE_GPU_FLAG else "--no-gpu"
video_matching = "sequential"

# تشغيل معالجة إطارات الفيديو باستخدام ns-process-data images مع المطابقة المتسلسلة (sequential)
!ns-process-data images \
    --data "{IMAGES_DIR}" \
    --output-dir "{PROCESSED_DATA_DIR}" \
    --camera-type {CAMERA_TYPE} \
    --matching-method {video_matching} \
    --sfm-tool {SFM_TOOL} \
    {gpu_flag}

# بديل قديم معلق للأمر المباشر (ns-process-data video):
# !ns-process-data video --data "{VIDEO_PATH}" --output-dir "{PROCESSED_DATA_DIR}" --num-frames-target 300 {gpu_flag}

## 🧪 [اختياري - OPTIONAL] مسار hloc المتقدم (SuperPoint + LightGlue)
مسار بديل مخصص للمشاهد المعقدة أو منخفضة التباين باستخدام خوارزميات التعلم العميق لاستخراج المعالم ومطابقتها.

In [ ]:
# ==============================================================================
# 🧪 [اختياري - OPTIONAL] مسار SfM البديل عبر مكتبة hloc المتقدمة
# (SuperPoint + LightGlue)
# ملاحظة تقنية وتوافقية:
# حزمة pycolmap ومكتبة hloc تعتمدان على امتدادات C++ قد تواجه صعوبة في التثبيت
# في بيئات Python 3.13 أو إصدارات PyTorch الأحدث لعدم توفر عجلات مسبقة البناء (Prebuilt Wheels).
# في حال فشل التثبيت، لن يتأثر الدفتر، ويمكنك الاعتماد على مسار COLMAP القياسي.
# ==============================================================================
import os
import shutil
import subprocess
from pathlib import Path

# تنظيف المجلد السابق قبل التجربة
if Path(PROCESSED_DATA_DIR).exists():
    shutil.rmtree(PROCESSED_DATA_DIR)

gpu_flag = "" if USE_GPU_FLAG else "--no-gpu"

hloc_install_cmd = """
pip install pycolmap && \
if [ ! -d "Hierarchical-Localization" ]; then
    git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization/
fi && \
pip install --quiet -e Hierarchical-Localization
"""

print("⏳ جاري محاولة تثبيت حزمة hloc (Hierarchical-Localization) و pycolmap...")
ret = subprocess.run(hloc_install_cmd, shell=True, capture_output=True, text=True)

if ret.returncode != 0:
    print("=" * 60)
    print("⚠️ تعذر تثبيت hloc بنجاح (قد يعود ذلك لعدم توافق بيئة Python 3.13 أو إصدارات PyTorch الحديثة).")
    print("💡 لا تقلق، لن يتأثر الدفتر: يمكنك استخدام مسار COLMAP الافتراضي في الخلايا السابقة.")
    print(f"تفاصيل الخطأ: {ret.stderr[:300]}")
    print("=" * 60)
else:
    print("✅ تم تثبيت hloc بنجاح! جاري تشغيل SfM باستخدام SuperPoint + LightGlue...")
    !ns-process-data images \
        --data "{IMAGES_DIR}" \
        --output-dir "{PROCESSED_DATA_DIR}" \
        --camera-type {CAMERA_TYPE} \
        --matching-method {MATCHING} \
        --sfm-tool hloc \
        --feature-type superpoint \
        --matcher-type superpoint+lightglue \
        {gpu_flag}

## 📊 فحص نسبة تسجيل الكاميرات (Registration Rate Evaluation)
قراءة ملف `transforms.json` الناتج عن COLMAP لحساب نسبة الصور المسجلة مقارنة بإجمالي الصور المرفوعة، مع تفعيل بوابة التحقق قبل التدريب.

In [ ]:
import json
from pathlib import Path

transforms_file = Path(PROCESSED_DATA_DIR) / "transforms.json"

# متغير البوابة للتحقق من جاهزية البيانات للتدريب
REGISTRATION_PASSED = False
registration_ratio = 0.0
registered_count = 0
uploaded_count = 0

# حساب عدد الصور المرفوعة في مجلد الصور
source_dir = Path(IMAGES_DIR)
valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
if source_dir.exists():
    uploaded_files = [p for p in source_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_exts]
    uploaded_count = len(uploaded_files)

if not transforms_file.exists():
    print("=" * 60)
    print(f"❌ خطأ فادح: لم يتم العثور على ملف {transforms_file}!")
    print("هذا يعني أن مرحلة استخراج مواضع الكاميرات (COLMAP/hloc) لم تكتمل بنجاح.")
    print("=" * 60)
    REGISTRATION_PASSED = False
else:
    with open(transforms_file, "r", encoding="utf-8") as f:
        transforms_data = json.load(f)

    frames = transforms_data.get("frames", [])
    registered_count = len(frames)

    if uploaded_count == 0:
        uploaded_count = registered_count

    registration_ratio = (registered_count / uploaded_count * 100) if uploaded_count > 0 else 0.0

    print("=" * 60)
    print("📊 تقرير تسجيل الكاميرات (Camera Registration Report)")
    print("=" * 60)
    print(f"📸 إجمالي الصور المرفوعة (Uploaded Images):    {uploaded_count}")
    print(f"✅ الصور المسجلة بنجاح (Registered Cameras):   {registered_count}")
    print(f"📈 نسبة التسجيل (Registration Percentage):      {registration_ratio:.1f}%")
    print("=" * 60)

    if registration_ratio >= 90.0:
        REGISTRATION_PASSED = True
        print(f"🟢 ممتاز جداً ({registration_ratio:.1f}% >= 90%): ترابط هندسي كامل بين الصور. المشهد جاهز تماماً للتدريب.")
    elif 70.0 <= registration_ratio < 90.0:
        REGISTRATION_PASSED = True
        print(f"🟡 جيد ({registration_ratio:.1f}% بين 70% و 89%): تم تسجيل معظم الصور، يمكنك المتابعة إلى التدريب (قد تلاحظ بعض الفجوات الطفيفة في الأماكن المعتمة).")
    else:
        REGISTRATION_PASSED = False
        print(f"🔴 توقف! نسبة التسجيل منخفضة وغير مقبولة ({registration_ratio:.1f}% < 70%):")
        print("COLMAP لم يتمكن من ربط 30% أو أكثر من الصور ببعضها، وسيتم إيقاف التدريب تلقائياً لتفادي تشوه المشهد.")
        print("\n📋 قائمة خطوات المعالجة والحلول المقترحة (مرتبة حسب الأولوية):")
        print("  1. التأكد من الصور الأصلية وبيانات EXIF: تأكد من عدم إرسال الصور عبر واتساب/تيليجرام كصور عادية مما يحذف بيانات البعد البؤري، واستخدم كابل USB أو Google Drive كملفات أصلية.")
        print("  2. استخدام المطابقة الشاملة: تأكد من ضبط MATCHING = 'exhaustive' في خلية المعايير لمقارنة جميع أزواج الصور بدقة.")
        print("  3. تجربة أداة hloc البديلة: تعيين SFM_TOOL = 'hloc' مع شبكة SuperPoint + LightGlue في الخلية الاختيارية لتجاوز ضعف المعالم.")
        print("  4. تصوير فيديو تجول بطيء: تسجيل فيديو بحركة مشي هادئة ومستمرة للشقة، أو التحرك بمقدار 30 إلى 50 سم فقط بين كل لقطة والأخرى مع ضمان تداخل بنسبة 70% إلى 80%.")
        print("  5. تجنب الجدران البيضاء المصمتة والدوران السريع في المكان: احرص دائماً على وجود أثاث أو معالم مميزة في إطار الكاميرا أثناء الحركة.")

## 5️⃣ تدريب نموذج Splatfacto (Train 3D Gaussian Splatting)
بدء تدريب النموذج باستخدام `ns-train splatfacto`.
> ⏱️ **تنبيه هام جداً حول استهلاك الذاكرة (RAM) والـ JIT Compile:**
> - تقوم مكتبة `gsplat` في أول تشغيل بتجميع مصرف CUDA عبر أداة `ninja`.
> - نحدد `MAX_JOBS=2` لمنع استهلاك رامات Colab بالكامل وتفادي خطأ الخروج المفاجئ (Exit code 137 OOM).
> - عملية التجميع تستغرق حوالي **3 إلى 4 دقائق** للمرة الأولى فقط، ثم يبدأ التدريب مباشرة.
> - **يرجى عدم إيقاف الخلية يدوياً!** اتركها تعمل حتى تظهر علامة الصح الخضراء ✅ معلنة اكتمال التدريب.

In [ ]:
# التحقق من بوابة نسبة التسجيل
if not globals().get('REGISTRATION_PASSED', False):
    raise RuntimeError(
        '❌ تم إيقاف التدريب تلقائياً! نسبة تسجيل الكاميرات أقل من 70% (أو لم تكتمل المعالجة بنجاح).\n'
        'يرجى مراجعة تقرير التسجيل وخطوات المعالجة المقترحة في الخلية السابقة قبل بدء التدريب.'
    )

import os

# تحديد عدد مهام التجميع بـ 2 لتفادي استهلاك رامات Colab بالكامل (OOM 137)
os.environ["MAX_JOBS"] = "2"

print(f"🚀 بدء تدريب Splatfacto لعدد {MAX_ITERATIONS} خطوة (مع MAX_JOBS=2)...")
print("⏳ في المرة الأولى، ستقوم gsplat بتجميع CUDA (حوالي 3-4 دقائق)، ثم يبدأ التدريب...")
print("يرجى الانتظار حتى تنتهي الخلية تماماً وتظهر علامة الصح الخضراء ✅")

# بدء التدريب مع MAX_JOBS=2 وحفظ دوري كل 1000 خطوة
!MAX_JOBS=2 ns-train splatfacto \
    --data "{PROCESSED_DATA_DIR}" \
    --output-dir "{TRAIN_OUTPUTS_DIR}" \
    --experiment-name "{EXPERIMENT_NAME}" \
    --max-num-iterations {MAX_ITERATIONS} \
    --steps-per-save 1000 \
    --viewer.quit-on-train-completion True


## 6️⃣ تصدير المشهد وتنزيله مباشرة (Export & Download .ply)
نقوم بتصدير المشهد بصيغة `.ply`، وحساب حجمه الفعلي بالميجابايت، ثم استدعاء دالة `google.colab.files.download()` لتنزيل الملف مباشرة إلى جهاز الكمبيوتر الخاص بك.
*(ملاحظة: يتضمن الكود إصلاحاً تلقائياً للتوافق مع تحديث الأمان في PyTorch 2.6+ لتفادي UnpicklingError)*

In [ ]:
import glob
import os
from pathlib import Path
from google.colab import files

# إصلاح التوافق مع PyTorch 2.6+ (weights_only=False)
try:
    import nerfstudio.utils.eval_utils as eval_utils
    eval_file = Path(eval_utils.__file__)
    content = eval_file.read_text(encoding="utf-8")
    if 'weights_only=False' not in content:
        content = content.replace(
            'torch.load(load_path, map_location="cpu")',
            'torch.load(load_path, map_location="cpu", weights_only=False)'
        )
        eval_file.write_text(content, encoding="utf-8")
        print("✅ تم تفعيل التوافق مع PyTorch 2.6+ بنجاح!")
except Exception as e:
    print(f"تنبيه التوافق: {e}")

# البحث التلقائي عن أحدث ملف config.yml الناتج عن التدريب
config_files = list(Path(TRAIN_OUTPUTS_DIR).glob(f"**/{EXPERIMENT_NAME}/**/config.yml"))
if not config_files:
    config_files = list(Path(TRAIN_OUTPUTS_DIR).glob("**/config.yml"))

if not config_files:
    raise FileNotFoundError(f"❌ لم يتم العثور على أي ملف config.yml في: {TRAIN_OUTPUTS_DIR}. تأكد من إتمام التدريب.")

latest_config = max(config_files, key=os.path.getmtime)
print(f"📄 Config file found: {latest_config}")

# مسار مجلد التصدير المؤقت واسم الملف
local_export_dir = Path(EXPORTS_DIR)
local_export_dir.mkdir(parents=True, exist_ok=True)
ply_filename = f"{EXPERIMENT_NAME}.ply"

# تنفيذ أمر تصدير الـ Gaussian Splat
!ns-export gaussian-splat \
    --load-config "{latest_config}" \
    --output-dir "{local_export_dir}" \
    --output-filename "{ply_filename}"

exported_file = local_export_dir / ply_filename

if exported_file.exists():
    file_size_bytes = exported_file.stat().st_size
    file_size_mb = file_size_bytes / (1024 * 1024)
    
    print("=" * 60)
    print("🎉 تم تصدير ملف الـ Gaussian Splat بنجاح!")
    print(f"📁 اسم الملف:              {ply_filename}")
    print(f"⚖️ حجم الملف (File Size):    {file_size_mb:.2f} MB")
    print(f"💾 المسار المحلي:           {exported_file}")
    print("=" * 60)
    print("⬇️ جاري بدء التنزيل المباشر إلى جهازك...")

    # بدء التحميل المباشر إلى جهاز الكمبيوتر
    files.download(str(exported_file))
else:
    print("❌ تعذر العثور على ملف .ply المصدر، يرجى مراجعة سجلات الخطأ أعلاه.")


## 🎯 الخطوات اللاحقة خارج Colab (Post-Processing & Viewing)
بعد اكتمال تنزيل الملف إلى جهازك:
1. **نقل الملف:** انقله إلى مجلد المشروع المحلي `maskan3d-lab/outputs/`.
2. **التنظيف والتحسين (Cleanup):**
   - افتح أداة **SuperSplat** (عبر المتصفح أو محلياً) لتحرير المشهد، قص الشوائب، وتعديل محور الأرضية.
3. **العرض على الجوال والويب (Web/Mobile Viewer):**
   - استخدم مكتبة **GaussianSplats3D** لتضمين الجولة التفاعلية في تطبيقك أو موقعك الإلكتروني مع دعم كامل وسلس لشاشات اللمس.
